In [17]:
from typing import List

In [18]:
class Node :
    """Défini un noeud de l'arbre"""

    def __init__(self, value : int = None):
        self.weight : int = 1
        self.value : int = value
        self.id : int = id

    def increment(self):
        self.weight += 1
    
    def setId(self, id : int):
        self.id = id
    
    #getters
    def getId(self):
        return self.id

    def getValue(self):
        return self.value
    
    def getWeight(self):
        return self.weight

    #str method
    def __str__(self) -> str:
        return "("+str(self.value)+":"+str(self.weight)+")"
    

In [ ]:
class HuffmanTree :
    def __init__(self):
        self.tab : List[Node] = []
        self.idLastAdded : int = -1 # indice dans tab du dernier node ajouté
        self.tab.append(Node(value="#"))
        self.idLastAdded = 0
        self.tab[0].setId(0)
    
    def searchValue(self, value : int) -> Node:
        """
        Cherche une valeur dans l'arbre
        - param : value, la valeur à chercher dans l'arbre
        - returns : Le noeud si valeur déjà présente, None sinon"""
        for node in self.tab:
            if (node.getValue() == value):
                return node
        return None
    
    def addValue(self, value : int) -> Node:
        """
        Ajoute une valeur à l'arbre. 
        \n Si valeur déjà présente : augmente son poids et renvoie le node correspondant. 
        \n Sinon : crée le node et le renvoie
        """
        found = self.searchValue(value)
        if found :
            found.increment() # weight++ si dans l'arbre
            return found
        else :
            node : Node = Node(value)
            self.idLastAdded += 1
            node.setId(self.idLastAdded)
            self.tab.append(node)
            return node
        
    def getLeft(self, node : Node) -> Node:
        """Renvoie le fils gauche du node si défini"""
        id : int = node.getId()
        child : int = 2*id+1
        if child < len(self.tab):
            return self.tab[child]
        return None
    
    def getRight(self, node : Node) -> Node:
        """Renvoie le fils droit du node si défini"""
        id : int = node.getId()
        child : int = 2*id+2
        if child < len(self.tab):
            return self.tab[child]
        return None
    
    def getParent(self, node : Node) -> Node:
        if self.isRoot(node):
            return None
        id = node.getId()
        return self.tab[(id-1)//2]

    def _getSubtreeNodes(self, node : Node) -> List[Node]:
        """
        Récupère tous les nœuds du sous-arbre enraciné à node (incluant node lui-même)
        Retourne une liste de tuples (node, id_relatif) où id_relatif est la position
        relative dans le sous-arbre (0 pour la racine du sous-arbre)
        """
        if node is None:
            return []
        
        result = [(node, 0)]  # Le nœud racine a l'id relatif 0
        queue = [(node, 0)]  # File pour parcours en largeur
        
        while queue:
            current_node, relative_id = queue.pop(0)
            
            # Fils gauche
            left = self.getLeft(current_node)
            if left is not None:
                left_relative_id = 2 * relative_id + 1
                result.append((left, left_relative_id))
                queue.append((left, left_relative_id))
            
            # Fils droit
            right = self.getRight(current_node)
            if right is not None:
                right_relative_id = 2 * relative_id + 2
                result.append((right, right_relative_id))
                queue.append((right, right_relative_id))
        
        return result

    def swapNodes(self, nodeA : Node, nodeB : Node):
        """
        Échange deux nœuds ainsi que toute leur descendance.
        Les sous-arbres complets sont échangés.
        """
        if nodeA is None or nodeB is None:
            return
        
        idA = nodeA.getId()
        idB = nodeB.getId()
        
        if idA == idB:
            return  # Même nœud, rien à faire
        
        # Récupérer tous les nœuds des deux sous-arbres avec leurs positions relatives
        subtreeA = self._getSubtreeNodes(nodeA)
        subtreeB = self._getSubtreeNodes(nodeB)
        
        # Créer des copies temporaires des sous-arbres
        tempA = [(node, idA + rel_id) for node, rel_id in subtreeA]
        tempB = [(node, idB + rel_id) for node, rel_id in subtreeB]
        
        # Échanger les nœuds dans le tableau
        # D'abord, placer le sous-arbre B à la position A
        for node, abs_id in tempB:
            new_id = idA + (abs_id - idB)
            if new_id < len(self.tab):
                self.tab[new_id] = node
                node.setId(new_id)
        
        # Ensuite, placer le sous-arbre A à la position B
        for node, abs_id in tempA:
            new_id = idB + (abs_id - idA)
            if new_id < len(self.tab):
                self.tab[new_id] = node
                node.setId(new_id)
    
    def isLeaf(self, node : Node) -> bool:
        return ((self.getLeft(node) is None) and (self.getRight(node) is None))

    def isRoot(self, node : Node) -> bool:
        id : int = node.getId()
        return id == 0

    def _printTree(self, node: Node, prefix: str = "", isLeft: bool = True) -> str:
        """Méthode récursive pour afficher l'arbre de manière visuelle"""
        if node is None:
            return ""
        
        result = ""
        
        # Afficher le noeud actuel
        result += prefix
        result += "├── " if isLeft else "└── "
        result += f"[{node.getId()}] {node.getValue()}:{node.getWeight()}\n"
        
        # Préparer le préfixe pour les enfants
        newPrefix = prefix + ("│   " if isLeft else "    ")
        
        # Obtenir les enfants
        left = self.getLeft(node)
        right = self.getRight(node)
        
        # Afficher les enfants s'ils existent
        if left is not None or right is not None:
            if left is not None:
                result += self._printTree(left, newPrefix, True)
            if right is not None:
                result += self._printTree(right, newPrefix, False)
        
        return result

    def __str__(self) -> str:
        if len(self.tab) == 0:
            return "Arbre vide"
        
        result = f"Arbre de Huffman (taille: {len(self.tab)})\n"
        result += f"[{self.tab[0].getId()}] {self.tab[0].getValue()}:{self.tab[0].getWeight()}\n"
        
        # Afficher les enfants de la racine
        left = self.getLeft(self.tab[0])
        right = self.getRight(self.tab[0])
        
        if left is not None:
            result += self._printTree(left, "", True)
        if right is not None:
            result += self._printTree(right, "", False)
        
        return result

In [29]:
# Test de l'affichage de l'arbre
tree = HuffmanTree()

# Ajouter quelques valeurs pour tester
tree.addValue('A')
tree.addValue('B')
tree.addValue('C')
tree.addValue('D')
tree.addValue('E')
tree.addValue('A')  # Incrémente le poids de A

# Afficher l'arbre
print(tree)

Arbre de Huffman (taille: 6)
[0] #:1
├── [1] A:2
│   ├── [3] C:1
│   └── [4] D:1
└── [2] B:1
    ├── [5] E:1



In [31]:
# Test de la méthode swapNodes
print("=== Test de swapNodes ===\n")

# Créer un arbre avec plusieurs nœuds
tree = HuffmanTree()
tree.addValue('A')
tree.addValue('B')
tree.addValue('C')
tree.addValue('D')
tree.addValue('E')
tree.addValue('F')

print("Arbre initial:")
print(tree)
print("\n" + "="*50 + "\n")

# Échanger les nœuds d'ID 1 et 2 (les deux enfants de la racine)
nodeA = tree.tab[1]
nodeB = tree.tab[5]

print(f"Échange des nœuds [{nodeA.getId()}] {nodeA.getValue()} et [{nodeB.getId()}] {nodeB.getValue()}\n")
tree.swapNodes(nodeA, nodeB)

print("Arbre après swap:")
print(tree)

=== Test de swapNodes ===

Arbre initial:
Arbre de Huffman (taille: 7)
[0] #:1
├── [1] A:1
│   ├── [3] C:1
│   └── [4] D:1
└── [2] B:1
    ├── [5] E:1
    └── [6] F:1



Échange des nœuds [1] A et [5] E

Arbre après swap:
Arbre de Huffman (taille: 7)
[0] #:1
├── [1] A:1
│   ├── [3] C:1
│   └── [4] D:1
└── [2] B:1
    ├── [5] E:1
    └── [6] F:1

